# Red neuronal artificial para reconocer el tipo de un pokemon

En esta tarea ustedes deben diseñar, entrenar y evaluar un modelo de red neuronal con arquitectura convolucional para resolver el problema de reconocer el tipo de un pokemon en base a una imagen del mismo y a sus atributos. El principal desafio es el desbalance y poca cantidad de ejemplos en el dataset. 

**RECONOCER EL TIPO DE UN POKEMON EN BASE A LA IMAGEN Y SUS ATRIBUTOS**

A continuación se instancia el dataset y se itera para presentar algunos ejemplos:

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import matplotlib.pyplot as plt
from pokemon_utils import PokemonImages

dataset = PokemonImages('data/')

fig, ax = plt.subplots(3, 3, figsize=(7, 5), tight_layout=True)
for ax_, (image, label, name, attributes) in zip(ax.ravel(), dataset):
    ax_.imshow(image.permute(-2, -1, 0))
    ax_.axis('off');

Cada ejemplo tiene su imagen, su etiqueta, su nombre y sus atributos:

In [ ]:
image, label, name, attributes = dataset[0]
type(image), image.shape, label, name, attributes

Se puede obtener el nombre de la clase con:

In [ ]:
dataset.categories[label]

Y los atributos disponibles son:

In [ ]:
dataset.attribute_names

La cantidad de ejemplos por clase es:

In [ ]:
from collections import Counter
import pandas as pd
x = Counter(dataset.labels)
for key in sorted(x):
    print(f"{dataset.categories[key]}: {x[key]}")

En lo que sigue utilice los siguientes conjuntos de entrenamiento (train) y prueba (test). 

In [ ]:
import numpy as np
from torch.utils.data import Subset, DataLoader
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(np.arange(len(dataset)),test_size=0.15, random_state=1234, 
                                       shuffle=True, stratify=dataset.labels)

train_dataset = Subset(dataset, train_idx)
test_dataset = Subset(dataset, test_idx)

### Definición del modelo

Defina e implemente un modelo apropiado para el problema. Justifique sus decisiones de diseño.

## Justificación
Para abordar el problema de reconocer usando el tipo con los atributos y la imagen sale más factible usar por un lado una arquitectura CNN para la imagen, y una arquitectura MLP para los atributos.

### CNN (imagen)
En este caso cada imagen es de 3x256x256, lo que tiene un total de 196,608 valores, en una MLP se necesitaría esa cantidad de neuronas en primera capa, lo que es inmanejable. Además como se vió en el curso las CNN son especiales para este tipo de problemas de reconocer patrones en imagenes.

Ahora la arquitectura de este modelo será la siguiente.
Se utilizarán 4 bloques convulcionales(conv2D->Relu->Maxpool2D(2)), esto se hace para que se pueda comprimir lo maximo posible la matriz, pasando de 256x256 a 16x16 de forma gradual dejando un vector de features más conveniente

Se usará un maxpool2D de tamaño 2 porque MaxPool(2) descarta 3 de cada 4 valores, MaxPool(4) descarta 15 de 16. Con un dataset pequeño cada píxel de información cuenta, así que conviene reducir suavemente. Además las capas conv intermedias tienen la oportunidad de aprender features antes de cada compresión.

Se utiliza AdaptiveAvgPool2d(4,4) al final de los bloques para forzar la salida a un tamaño fijo de 128×4×4 = 2048, haciendo el modelo robusto a variaciones de tamaño y reduciendo el vector antes del clasificador.


### MLP (atributos)
Los 6 atributos numéricos (HP, Attack, Defense, Speed, Height, Weight) pasan por una capa Linear(6→32) + ReLU. Simple porque los atributos ya son valores limpios y pocos.
Fusión
Ambas salidas se concatenan (2048 + 32 = 2080) y pasan por un clasificador Linear(2080→256→18) con Dropout(0.5) para regularizar, dado que el dataset es pequeño (~600 imágenes para 18 clases) y el riesgo de overfitting es alto.


In [ ]:


import torch
import torch.nn as nn

class PokemonCNN(nn.Module):

    def __init__(self, num_classes=18, num_attributes=6):
        super().__init__()

        # Rama imagen: 4 bloques Conv → ReLU → MaxPool2d(2)
        # Cada MaxPool divide el spatial por 2: 256→128→64→32→16
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),   # 3×256×256 → 16×256×256
            nn.ReLU(),
            nn.MaxPool2d(2),                               # → 16×128×128

            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # → 32×128×128
            nn.ReLU(),
            nn.MaxPool2d(2),                               # → 32×64×64

            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # → 64×64×64
            nn.ReLU(),
            nn.MaxPool2d(2),                               # → 64×32×32

            nn.Conv2d(64, 128, kernel_size=3, padding=1), # → 128×32×32
            nn.ReLU(),
            nn.MaxPool2d(2),                               # → 128×16×16
        )

        # Fuerza salida a 4×4 sin importar el spatial que llegue
        # 128×16×16 → 128×4×4 → flatten → 2048
        self.pool = nn.AdaptiveAvgPool2d((4, 4))

        # Rama atributos: 6 números → 32
        self.attr_mlp = nn.Sequential(
            nn.Linear(num_attributes, 32),
            nn.ReLU(),
        )

        # Clasificador final: concatena CNN (2048) + atributos (32) → 18 clases
        self.classifier = nn.Sequential(
            nn.Linear(2048 + 32, 256),
            nn.ReLU(),
            nn.Dropout(0.5),      # regularización: dataset pequeño, riesgo alto de overfit
            nn.Linear(256, num_classes),
        )

    def forward(self, image, attributes):
        # Rama imagen
        z = self.features(image)       # → 128×16×16
        z = self.pool(z)               # → 128×4×4
        z = z.view(z.size(0), -1)      # flatten → 2048

        # Rama atributos
        a = self.attr_mlp(attributes)  # → 32

        # Fusión
        out = torch.cat([z, a], dim=1) # → 2080
        return self.classifier(out)    # → 18 logits


model = PokemonCNN(num_classes=len(dataset.categories), num_attributes=len(dataset.attribute_names))
print(model)


### Entrenamiento del modelo

Entrene el modelo y muestre las curvas de aprendizaje. Justifique la elección de hiperparámetros.

In [ ]:
from torch.utils.data import random_split, DataLoader
import torch.optim as optim


In [ ]:
## Split train/test con estratificación

# 1. Obtenemos las etiquetas de todo el dataset
todas_las_etiquetas = [label for _, label, _, _ in dataset]

# 2. Generamos índices
indices = list(range(len(dataset)))

# 3. Usamos train_test_split con stratify para mantener la proporción de clases
train_idx, test_idx = train_test_split(
    indices, 
    test_size=0.2,           # 20% para prueba, 80% para entrenamiento
    random_state=42,         # Para reproducibilidad
    stratify=todas_las_etiquetas  # ¡Mantiene la proporción de clases!
)

# 4. Creamos los subconjuntos usando los índices estratificados
train_set = Subset(dataset, train_idx)
test_set = Subset(dataset, test_idx)

# 5. Pasamos a los DataLoaders
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False)


In [ ]:
## Modelo, loss, optimizer
# Detecta CUDA (Nvidia), MPS (Apple Silicon) o cae de vuelta en CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Usando el dispositivo: {device}")

model = PokemonCNN(num_classes=len(dataset.categories),
                    num_attributes=len(dataset.attribute_names)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-2, weight_decay=1e-4)

In [ ]:
## Training loop
def train(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct = 0.0, 0

    for images, labels, _, attributes in loader:
        images     = images.to(device)
        labels     = labels.to(device)
        attributes = attributes.to(device)

        optimizer.zero_grad()
        outputs = model(images, attributes)   # forward con ambas entradas
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(dim=1) == labels).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0.0, 0

    with torch.no_grad():
        for images, labels, _, attributes in loader:
            images     = images.to(device)
            labels     = labels.to(device)
            attributes = attributes.to(device)

            outputs = model(images, attributes)
            loss    = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            correct    += (outputs.argmax(dim=1) == labels).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)


In [ ]:
# Ejecutar épocas CON AUGMENTATION
num_epochs = 30
train_losses, test_losses = [], []
train_accs,   test_accs   = [], []
best_test_acc = 0.0

# Reinicializar el modelo
model = PokemonCNN(num_classes=len(dataset.categories),
                    num_attributes=len(dataset.attribute_names)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-2, weight_decay=1e-4)

for epoch in range(num_epochs):
    # Usar los DataLoaders con augmentation
    tr_loss, tr_acc = train(model, train_loader_aug, criterion, optimizer, device)
    te_loss, te_acc = evaluate(model, test_loader_aug, criterion, device)

    # Guardar si es el mejor hasta ahora
    if te_acc > best_test_acc:
        best_test_acc = te_acc
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  ✓ Mejor modelo guardado (test acc: {te_acc:.3f})")

    # Guardar curvas
    train_losses.append(tr_loss)
    test_losses.append(te_loss)
    train_accs.append(tr_acc)
    test_accs.append(te_acc)

    print(f"Época {epoch+1:02d}/{num_epochs} | "
          f"Train loss: {tr_loss:.4f} acc: {tr_acc:.3f} | "
          f"Test loss: {te_loss:.4f} acc: {te_acc:.3f}")

print(f"\nMejor test accuracy CON AUGMENTATION: {best_test_acc:.3f}")

# Cargar el mejor modelo antes de evaluar
model.load_state_dict(torch.load("best_model.pth"))


In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, label='Train')
ax1.plot(test_losses,  label='Test')
ax1.set_title('Loss')
ax1.set_xlabel('Época')
ax1.legend()

ax2.plot(train_accs, label='Train')
ax2.plot(test_accs,  label='Test')
ax2.set_title('Accuracy')
ax2.set_xlabel('Época')
ax2.legend()

plt.tight_layout()
plt.show()

### Evaluación del modelo

Evalúe el modelo en el conjunto de prueba usando matriz de confusión y reporte de clasificación. Discuta los resultados.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

def evaluar_modelo(model, test_loader, device, class_names):
    # 1. Poner el modelo en modo evaluación (apaga el Dropout y Batch Norm)
    model.eval()
    
    all_preds = []
    all_labels = []
    
    # 2. Desactivar el cálculo de gradientes para ahorrar memoria y acelerar
    with torch.no_grad():
        # Nota: Ajusta cómo desempaquetas el batch según tu DataLoader
        # Asumiendo que devuelve (imagenes, labels_categorias, _, atributos)
        for images, labels_cat, _, attributes in test_loader:
            images = images.to(device)
            labels_cat = labels_cat.to(device)
            attributes = attributes.to(device)
            
            # Pasar imágenes y atributos por el modelo
            out_cat = model(images, attributes)
            
            # Obtener el índice de la probabilidad más alta
            _, preds_cat = torch.max(out_cat, 1)
            
            # Mover a CPU y convertir a numpy para usar con scikit-learn
            all_preds.extend(preds_cat.cpu().numpy())
            all_labels.extend(labels_cat.cpu().numpy())
            
    # --- REPORTE DE CLASIFICACIÓN ---
    print("\n" + "="*50)
    print("REPORTE DE CLASIFICACIÓN (Categorías)")
    print("="*50)
    reporte = classification_report(all_labels, all_preds, target_names=class_names)
    print(reporte)
    
    # --- MATRIZ DE CONFUSIÓN ---
    cm = confusion_matrix(all_labels, all_preds)
    
    plt.figure(figsize=(10, 8)) # Ajusta el tamaño si tienes muchas clases
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    
    plt.xlabel('Predicción del Modelo', fontsize=12, fontweight='bold')
    plt.ylabel('Etiqueta Real', fontsize=12, fontweight='bold')
    plt.title('Matriz de Confusión - Reconocimiento de Pokémon', fontsize=14, fontweight='bold')
    
    # Rotar las etiquetas del eje X para que se lean bien
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()





### Análisis de errores

Analice algunos ejemplos mal clasificados y comente lo que observa.

### Aumentación de datos

Implemente un esquema de aumentación aleatoria de datos y compare los resultados con y sin aumentación.

In [ ]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Crear un wrapper que aplique transformaciones
class PokemonDatasetWithAugmentation:
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        image, label, name, attributes = self.dataset[idx]
        
        # image es un tensor de torch de shape (3, 256, 256)
        # Convertir a PIL para aplicar transformaciones
        if self.transform:
            image = transforms.ToPILImage()(image)
            image = self.transform(image)
        
        return image, label, name, attributes

# Transformaciones SOLO para entrenamiento (augmentation)
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),           # Espejo horizontal
    transforms.RandomRotation(degrees=15),             # Rotación ±15°
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # Variar luz y contraste
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),  # Pequeñas traslaciones
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Sin transformaciones para test (solo normalización)
test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Aplicar aumentación al train_set
train_set_augmented = PokemonDatasetWithAugmentation(train_set, transform=train_transforms)
test_set_augmented = PokemonDatasetWithAugmentation(test_set, transform=test_transforms)

# Crear nuevos DataLoaders con augmentation
train_loader_aug = DataLoader(train_set_augmented, batch_size=32, shuffle=True)
test_loader_aug = DataLoader(test_set_augmented, batch_size=32, shuffle=False)

print("✓ DataLoaders con augmentation creados")
print(f"Train set: {len(train_set_augmented)} imágenes")
print(f"Test set: {len(test_set_augmented)} imágenes")

# Ahora entrena con: train_loader_aug y test_loader_aug
# En lugar de train_loader y test_loader
